# 09a — Figure 2/3 Published-Loss Audit

**Question:** Why does the printed Figure 2 equation give a different covariance-weighted loss from Table IV? This notebook isolates formula rounding, the DESI error convention, the Pantheon+/DESY5 covariance, and the residual domain. It does **not** rerun PySR or overwrite Notebook 09.

Every execution creates a new, uniquely named directory under `results/09a_fig23_loss_audit/`. The paper's displayed formulas are rounded transcriptions, not raw PySR constants; a numerical match to a reported loss is evidence, not proof of the authors' hidden constants or covariance.

## 1. Load frozen inputs and audit specifications

Notebook 08 supplies the selected data and transformed full covariance. The audit specification records the expressions and losses printed in the paper's Tables IV–V and Eqs. 12–13. We retain the exact printed decimal strings so any rounding sensitivity is visible.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'configs/fig23_loss_audit.yaml').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loading import load_standard_dataset, with_desi_error_version
from src.loss_audit import (evaluate_expression, evaluate_formula_bank, block_chi2,
    diagonal_sn_loss, amplitude_quadratic, amplitude_loss, magnitude_space_diagnostic)

CONFIG_PATH = PROJECT_ROOT / 'configs/fig23_loss_audit.yaml'
CONFIG = yaml.safe_load(CONFIG_PATH.read_text())
SCENARIOS = tuple(CONFIG['formulas'])
DATA_PATHS = {name: PROJECT_ROOT / CONFIG['prepared_data_root'] / f'{name}.npz'
              for name in SCENARIOS}
DATA = {name: load_standard_dataset(path) for name, path in DATA_PATHS.items()}
VARIANTS = {(name, version): with_desi_error_version(DATA[name], version)
            for name in SCENARIOS for version in ('public_release', 'paper_table')}
print('Scenarios:', SCENARIOS, '| variants:', len(VARIANTS))

Scenarios: ('fig2_pantheon', 'fig3_desy5') | variants: 4


## 2. Verify the controlled comparison and create a new experiment

The Table I sensitivity case changes only six DESI variances. It is **not** the authors' unpublished fitted covariance. Store the hashes and preparation metadata so this audit can be traced back to the exact local inputs.

In [2]:
for name in SCENARIOS:
    public, table = VARIANTS[name, 'public_release'], VARIANTS[name, 'paper_table']
    np.testing.assert_array_equal(public['z'], table['z'])
    np.testing.assert_array_equal(public['y'], table['y'])
    np.testing.assert_array_equal(public['covariance'][6:, 6:], table['covariance'][6:, 6:])
    assert np.count_nonzero(public['covariance'][:6, 6:]) == 0
    assert np.count_nonzero(table['covariance'][:6, 6:]) == 0

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EXPERIMENT_ID = f'fig23_loss_audit_{stamp}_{uuid4().hex[:8]}'
EXPERIMENT_DIR = PROJECT_ROOT / CONFIG['output_root'] / EXPERIMENT_ID
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
MANIFEST = {'experiment_id': EXPERIMENT_ID, 'created_utc': datetime.now(timezone.utc).isoformat(),
            'status': 'started', 'paper_url': CONFIG['paper_url'], 'config': CONFIG,
            'config_sha256': hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),
            'prepared_inputs': {name: {'path': str(path),
                'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
                'raw_source_sha256': DATA[name]['metadata']['source_sha256']}
                for name, path in DATA_PATHS.items()}}
(EXPERIMENT_DIR / 'experiment.json').write_text(json.dumps(MANIFEST, indent=2) + '\n')
INPUTS = pd.DataFrame([{'scenario': name, 'n_total': len(data['y']),
    'n_sn': len(data['y']) - 6, 'sn_z_min': float(np.min(data['z'][6:])),
    'sn_z_max': float(np.max(data['z'][6:])),
    'sn_observable_kind': data['metadata']['config']['supernova']['observable_kind'],
    'calibration': json.dumps(data['metadata']['config']['supernova']['calibration'])}
    for name, data in DATA.items()])
INPUTS.to_csv(EXPERIMENT_DIR / 'input_manifest.csv', index=False)
display(INPUTS)
print('New experiment:', EXPERIMENT_DIR)

,scenario,n_total,n_sn,sn_z_min,sn_z_max,sn_observable_kind,calibration
0,fig2_pantheon,1596,1590,0.01016,2.26137,corrected_magnitude,"{""M"": -19.253, ""rd_mpc"": 138.28, ""rd_sigma_mpc..."
1,fig3_desy5,1760,1754,0.02509,1.12132,distance_modulus,"{""H0"": 70, ""rd_mpc"": 144.2857142857143, ""rd_si..."


New experiment: /Users/kemingliu/Desktop/Dark Energy/results/09a_fig23_loss_audit/fig23_loss_audit_20260920T045157127883Z_09f5ee72


## 3. Recompute losses for several printed paper candidates

A discrepancy for only one printed formula can be caused by coefficient rounding. Discrepancies across several structurally different formulas indicate an additional input or likelihood mismatch. All losses below use the **same prepared central values** and each named covariance version.

In [3]:
rows = []
for (name, version), data in VARIANTS.items():
    for result in evaluate_formula_bank(data, CONFIG['formulas'][name], data['covariance']):
        rows.append({'scenario': name, 'desi_error_version': version,
                     'n': len(data['y']), **result})
FORMULA_LOSSES = pd.DataFrame(rows)
FORMULA_LOSSES.to_csv(EXPERIMENT_DIR / 'formula_loss_comparison.csv', index=False)
display(FORMULA_LOSSES.loc[FORMULA_LOSSES.desi_error_version.eq('public_release'),
       ['scenario', 'formula_id', 'paper_reported_loss', 'our_loss', 'loss_difference']])

,scenario,formula_id,paper_reported_loss,our_loss,loss_difference
0,fig2_pantheon,eq12_printed,1.70,4.729393,3.029393
1,fig2_pantheon,table_iv_selected_literal,1.70,4.788842,3.088842
2,fig2_pantheon,table_iv_complexity_3,4.65,3.614543,-1.035457
3,fig2_pantheon,table_iv_complexity_5,2.69,1.874351,-0.815649
4,fig2_pantheon,table_iv_complexity_6,2.39,1.775713,-0.614287
5,fig2_pantheon,table_iv_complexity_9,1.68,0.923924,-0.756076
6,fig2_pantheon,table_iv_complexity_10,1.68,0.910111,-0.769889
14,fig3_desy5,eq13_printed,2.23,1.016472,-1.213528
15,fig3_desy5,table_v_selected_literal,2.23,0.989677,-1.240323
16,fig3_desy5,table_v_complexity_3,22.50,6.916685,-15.583315


## 4. Is the printed Figure 2 denominator precise enough?

Table IV writes the selected candidate as $z \times 0.76^z/0.03$ and reports loss about 1.70. A true denominator near 0.032 would also be displayed as 0.03 at two decimals. This scan is a **rounding sensitivity test**, not an estimate of the authors' hidden constant. We obtain the loss curve exactly from a generalized least-squares quadratic, without fitting PySR again.

In [4]:
scan_cfg = CONFIG['denominator_scan']
denominators = np.linspace(scan_cfg['minimum'], scan_cfg['maximum'], scan_cfg['points'])
scan_rows, anchor_rows = [], []
for version in ('public_release', 'paper_table'):
    data = VARIANTS['fig2_pantheon', version]
    shape = data['z'] * scan_cfg['base'] ** data['z']
    quad = amplitude_quadratic(data, data['covariance'], shape)
    losses = amplitude_loss(quad, 1 / denominators)
    scan_rows.extend({'desi_error_version': version, 'denominator': float(d),
                      'amplitude': float(1 / d), 'loss': float(loss)}
                     for d, loss in zip(denominators, losses))
    for d in (0.030, scan_cfg['illustrative_value'], 0.034, 1 / quad['best_amplitude']):
        anchor_rows.append({'desi_error_version': version, 'denominator': float(d),
                            'amplitude': float(1 / d),
                            'loss': float(amplitude_loss(quad, 1 / d)),
                            'gls_best_denominator': float(1 / quad['best_amplitude']),
                            'gls_best_loss': float(quad['best_loss'])})
SCAN = pd.DataFrame(scan_rows)
ANCHORS = pd.DataFrame(anchor_rows)
SCAN.to_csv(EXPERIMENT_DIR / 'denominator_scan.csv', index=False)
ANCHORS.to_csv(EXPERIMENT_DIR / 'denominator_anchors.csv', index=False)
fig, ax = plt.subplots(figsize=(7.5, 4.7))
for version, group in SCAN.groupby('desi_error_version', sort=False):
    ax.plot(group.denominator, group.loss, label=version)
ax.axhline(1.70, color='black', linestyle='--', label='Table IV reported loss 1.70')
for d, color in ((0.030, 'firebrick'), (0.032, 'darkorange')):
    ax.axvline(d, color=color, linestyle=':', label=f'denominator = {d:.3f}')
ax.set(xlabel='Denominator in z × 0.76^z / denominator',
       ylabel=r'Full-covariance $\chi^2/N$', title='Figure 2 coefficient-precision audit')
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / 'fig2_denominator_scan.png', dpi=180)
plt.close(fig)
display(ANCHORS)

,desi_error_version,denominator,amplitude,loss,gls_best_denominator,gls_best_loss
0,public_release,0.030000,33.333333,4.788842,0.033840,0.901680
1,public_release,0.032000,31.250000,1.686104,0.033840,0.901680
2,public_release,0.034000,29.411765,0.906933,0.033840,0.901680
3,public_release,0.033840,29.550818,0.901680,0.033840,0.901680
4,paper_table,0.030000,33.333333,4.814583,0.033837,0.902034
5,paper_table,0.032000,31.250000,1.690148,0.033837,0.902034
6,paper_table,0.034000,29.411765,0.907544,0.033837,0.902034
7,paper_table,0.033837,29.553617,0.902034,0.033837,0.902034


## 5. Separate covariance, residual-domain, and calibration questions

The six BAO observations and the SN block are independent in the prepared covariance, so their chi-square contributions can be separated exactly. The diagonal-SN and original-magnitude-domain calculations are **counterfactual diagnostics**, not alternate claims about what the paper used. Profiling a common magnitude offset quantifies calibration sensitivity but must not be treated as an authorized correction or part of the published fit.

In [5]:
diagnostic_rows = []
selected_ids = {'fig2_pantheon': 'eq12_printed', 'fig3_desy5': 'eq13_printed'}
for (name, version), data in VARIANTS.items():
    selected = next(item for item in CONFIG['formulas'][name]
                    if item['id'] == selected_ids[name])
    prediction = evaluate_expression(selected['expression'], data['z'])
    parts = block_chi2(data, prediction, data['covariance'])
    calibration = data['metadata']['config']['supernova']['calibration']
    mag = magnitude_space_diagnostic(data, prediction, calibration)
    diagnostic_rows.append({'scenario': name, 'desi_error_version': version,
        'formula_id': selected['id'], 'paper_reported_loss': selected['paper_reported_loss'],
        **parts, 'sn_fraction_of_chi2': parts['sn_chi2'] / parts['total_chi2'],
        'diagonal_sn_loss': diagonal_sn_loss(data, prediction, data['covariance']),
        **mag, 'magnitude_space_total_loss':
        (parts['bao_chi2'] + mag['sn_observable_chi2']) / len(data['y']),
        'profiled_magnitude_space_total_loss':
        (parts['bao_chi2'] + mag['profiled_sn_chi2']) / len(data['y'])})
DIAGNOSTICS = pd.DataFrame(diagnostic_rows)
DIAGNOSTICS.to_csv(EXPERIMENT_DIR / 'covariance_diagnostics.csv', index=False)
display(DIAGNOSTICS[['scenario', 'desi_error_version', 'total_loss',
    'bao_chi2', 'sn_chi2', 'diagonal_sn_loss', 'magnitude_space_total_loss',
    'mean_observable_residual', 'profiled_common_offset_mag']])

,scenario,desi_error_version,total_loss,bao_chi2,sn_chi2,diagonal_sn_loss,magnitude_space_total_loss,mean_observable_residual,profiled_common_offset_mag
0,fig2_pantheon,public_release,4.729393,526.200540,7021.910219,4.811578,4.024929,0.244500,-0.252516
1,fig2_pantheon,paper_table,4.754840,566.814473,7021.910219,4.837025,4.050376,0.244500,-0.252516
2,fig3_desy5,public_release,1.016472,16.704486,1772.286450,0.940887,0.975484,0.014974,-0.030451
3,fig3_desy5,paper_table,1.017151,17.898970,1772.286450,0.941566,0.976162,0.014974,-0.030451


## 6. Evidence-based interpretation and handoff

A denominator that numerically reproduces 1.70 does **not** reveal the authors' exact PySR constant: another covariance or data vector could shift the loss curve. Cross-check independent published candidates before concluding that rounding alone explains the mismatch. The next decisive external artifact is the authors' unrounded hall-of-fame expression together with the exact fit vector and covariance after the magnitude-to-distance transformation. Do not tune the covariance to force agreement.

In [6]:
def result(name, formula_id, version='public_release'):
    row = FORMULA_LOSSES.loc[(FORMULA_LOSSES.scenario == name)
        & (FORMULA_LOSSES.formula_id == formula_id)
        & (FORMULA_LOSSES.desi_error_version == version)]
    assert len(row) == 1
    return float(row.iloc[0].our_loss)

eq12 = result('fig2_pantheon', 'eq12_printed')
literal = result('fig2_pantheon', 'table_iv_selected_literal')
illustrative = float(ANCHORS.loc[(ANCHORS.desi_error_version == 'public_release')
    & np.isclose(ANCHORS.denominator, 0.032)].iloc[0].loss)
independent = result('fig2_pantheon', 'table_iv_complexity_10')
eq13 = result('fig3_desy5', 'eq13_printed')
fig2_diag = DIAGNOSTICS.loc[(DIAGNOSTICS.scenario == 'fig2_pantheon')
    & (DIAGNOSTICS.desi_error_version == 'public_release')].iloc[0]
lines = [
    '# Figure 2/3 published-loss audit', '',
    f'Experiment ID: `{EXPERIMENT_ID}`.', '',
    '## Main numerical checks', '',
    f'- Printed Fig. 2 Eq. 12: loss {eq12:.6f}; Table IV reports about 1.70.',
    f'- Literal Table IV selected row with denominator 0.030: loss {literal:.6f}.',
    f'- Illustrative denominator 0.032, which also displays as 0.03 to two decimals: loss {illustrative:.6f}. This is not a recovered author constant.',
    f'- Independent Table IV complexity-10 expression: loss {independent:.6f}; paper reports about 1.68.',
    f'- Printed Fig. 3 Eq. 13: loss {eq13:.6f}; Table V reports about 2.23.',
    f'- Fig. 2 printed Eq. 12: BAO chi-square {fig2_diag.bao_chi2:.3f}; SN chi-square {fig2_diag.sn_chi2:.3f}.',
    '', '## Interpretation', '',
    'The selected Fig. 2 formula is highly sensitive to the denominator rounded to 0.03 in the paper. A value near 0.032 can reproduce its reported loss under our inputs, but this numerical coincidence does not prove the hidden PySR coefficient.',
    'Other printed candidates also have loss discrepancies, so rounding alone has not established full reproduction of the authors\' data vector and covariance.',
    'Changing only six DESI variances or dropping SN off-diagonal correlations does not explain the printed Eq. 12 discrepancy.',
    'Next: request or locate the authors\' unrounded PySR hall-of-fame output, transformed SN vector, covariance, and exact loss implementation. Until then, keep the data/covariance and formula-precision hypotheses separate.',
    '', '## Artifacts', '',
    '`experiment.json`, `input_manifest.csv`, `formula_loss_comparison.csv`, `denominator_scan.csv`, `denominator_anchors.csv`, `fig2_denominator_scan.png`, and `covariance_diagnostics.csv`.',
    '', 'All calculations are read-only with respect to the prepared Notebook 08 data and the Notebook 09 fit results.'
]
(EXPERIMENT_DIR / 'findings.md').write_text('\n'.join(lines) + '\n')
assert np.isfinite(FORMULA_LOSSES.our_loss).all()
assert len(FORMULA_LOSSES) == sum(len(v) for v in CONFIG['formulas'].values()) * 2
MANIFEST['status'] = 'complete'
MANIFEST['completed_utc'] = datetime.now(timezone.utc).isoformat()
MANIFEST['artifact_names'] = sorted(p.name for p in EXPERIMENT_DIR.iterdir() if p.is_file())
(EXPERIMENT_DIR / 'experiment.json').write_text(json.dumps(MANIFEST, indent=2) + '\n')
print('Completed independent audit:', EXPERIMENT_DIR)
print('Fig. 2 printed / illustrative / independent-candidate losses:',
      round(eq12, 3), round(illustrative, 3), round(independent, 3))

Completed independent audit: /Users/kemingliu/Desktop/Dark Energy/results/09a_fig23_loss_audit/fig23_loss_audit_20260920T045157127883Z_09f5ee72
Fig. 2 printed / illustrative / independent-candidate losses: 4.729 1.686 0.91
